In [1]:
pip install pandas

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [2]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.metrics import classification_report
from sklearn.datasets import make_blobs
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GridSearchCV

from sklearn.metrics import accuracy_score


In [3]:
# Load the dataset
data = pd.read_csv("Titanic-Dataset.csv")
data.head(10)

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S
5,6,0,3,"Moran, Mr. James",male,NaN,0,0,330877,8.4583,NaN,Q
6,7,0,1,"McCarthy, Mr. Timothy J",male,54.0,0,0,17463,51.8625,E46,S
7,8,0,3,"Palsson, Master. Gosta Leonard",male,2.0,3,1,349909,21.0750,NaN,S
8,9,1,3,"Johnson, Mrs. Oscar W (Elisabeth Vilhelmina Berg)",female,27.0,0,2,347742,11.1333,NaN,S
9,10,1,2,"Nasser, Mrs. Nicholas (Adele Achem)",female,14.0,1,0,237736,30.0708,NaN,C


In [4]:
# Drop irrelevant columns (adjust as necessary)
data.drop(columns=["Name", "Ticket", "Cabin"], inplace=True, errors='ignore')
# Handling missing values
data = data.ffill()
# Encoding categorical features
label_encoders = {}
for col in ['Sex', 'Embarked']:
    if col in data.columns:
        le = LabelEncoder()
        data[col] = le.fit_transform(data[col].astype(str))
        label_encoders[col] = le
data.head(10)

,PassengerId,Survived,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked
0,1,0,3,1,22.0,1,0,7.2500,2
1,2,1,1,0,38.0,1,0,71.2833,0
2,3,1,3,0,26.0,0,0,7.9250,2
3,4,1,1,0,35.0,1,0,53.1000,2
4,5,0,3,1,35.0,0,0,8.0500,2
5,6,0,3,1,35.0,0,0,8.4583,1
6,7,0,1,1,54.0,0,0,51.8625,2
7,8,0,3,1,2.0,3,1,21.0750,2
8,9,1,3,0,27.0,0,2,11.1333,2
9,10,1,2,0,14.0,1,0,30.0708,0


In [5]:
# Assuming the Titanic dataset is loaded into the 'df' variable
X = data.drop(columns=['Survived'], errors='ignore')  # Features
y = data['Survived']  # Target variable
# Split the dataset into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)


In [6]:

# Define hyperparameter grid
param_grid = {'n_neighbors': list(range(1, 21))}  # Test k from 1 to 20

# Perform grid search
grid_search = GridSearchCV(KNeighborsClassifier(), param_grid, cv=5, scoring='accuracy')
grid_search.fit(X_train, y_train)

# Best k value
best_k = grid_search.best_params_['n_neighbors']
print(f"Best k: {best_k}")
# Train KNN with the best k
knn_best = KNeighborsClassifier(n_neighbors=best_k)
knn_best.fit(X_train, y_train)
y_pred_best = knn_best.predict(X_test)

# Evaluate accuracy
best_accuracy = accuracy_score(y_test, y_pred_best)
print(f"KNN Accuracy with best k ({best_k}): {best_accuracy:.2f}")


NameError: name 'GridSearchCV' is not defined

In [ ]:
# Range of k values from 1 to 20
k_values = list(range(1, 21))  
accuracy_scores = []

# Calculate accuracy for each k
for k in k_values:
    knn = KNeighborsClassifier(n_neighbors=k)
    knn.fit(X_train, y_train)  # Train on the training data
    y_pred = knn.predict(X_test)  # Predict on the test data    
    accuracy_scores.append(accuracy_score(y_test, y_pred))  # Append accuracy score

# Plotting accuracy vs. k
plt.figure(figsize=(8, 5))
plt.plot(k_values , accuracy_score, marker='o', linestyle='dashed', color='b')
plt.xlabel("Number of Neighbors (k)")
plt.ylabel("Accuracy")
plt.title("KNN Accuracy vs. Number of Neighbors")
plt.xticks(range(1, 21))  # Set x-axis ticks from 1 to 20
plt.grid(True)
plt.show()


In [ ]:
# Convert X_test to a DataFrame with proper column names
X_test_df = pd.DataFrame(data=X_test, columns=X.columns)
X_test_df["Prediction"] = y_pred

# Define color map manually
color_map = {0: "red", 1: "blue"}

# Create the scatter plot
plt.figure(figsize=(8, 6))
for label in sorted(X_test_df["Prediction"].unique()):
    subset = X_test_df[X_test_df["Prediction"] == label]
    plt.scatter(subset["Age"], subset["Fare"],
                label=f"{'Survived' if label == 1 else 'Not Survived'}",
                color=color_map[label], alpha=0.6)

plt.xlabel("Age (Standardized)")
plt.ylabel("Fare (Standardized)")
plt.title(f"KNN Survival Prediction Scatter Plot (k={best_k})")

# Custom legend
handles, labels = plt.gca().get_legend_handles_labels()
custom_labels = ['Not Survived', 'Survived']
plt.legend(handles, custom_labels, title="Predicted Survival")

plt.grid(True)
plt.tight_layout()
plt.show()